# Smart City-project van Nova Stad

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql.functions import col, lit, udf, pandas_udf, PandasUDFType
from pyspark.sql.types import *
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml import Pipeline
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.regression import GBTRegressor
from pyspark.ml.regression import DecisionTreeRegressor
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler, OneHotEncoder, Imputer
from pyspark.ml.classification import DecisionTreeClassifier, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator


### Data inladen

In [0]:
%sh
unzip -q /Volumes/workspace/default/course_files/bdd/bdd:.zip -d /Volumes/workspace/default/course_files/bdd/

In [0]:
display(dbutils.fs.ls("dbfs:/Volumes/workspace/default/course_files/bdd/bdd:/"))

In [0]:
# unzip images
dbutils.fs.cp(
    "dbfs:/Volumes/workspace/default/course_files/bdd/bdd100k_images_10k.zip",
    "file:/tmp/bdd_images.zip"
)

dbutils.fs.cp(
    "dbfs:/Volumes/workspace/default/course_files/bdd/bdd100k_labels.zip",
    "file:/tmp/bdd_labels.zip"
)

In [0]:
import zipfile

zip_path = "/Volumes/workspace/default/course_files/bdd100k_images_10k.zip"
extract_path = "/Volumes/workspace/default/course_files/bdd/"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

In [0]:
df_traffic = spark.read.csv("/FileStore/tables/metro_interstate_traffic_volume_train.csv", header=True, inferSchema=True)

In [0]:
df = spark.read.csv(
    "/Volumes/main/default/datasets/metro_interstate_traffic_volume_train.csv",
    header=True,
    inferSchema=True
)

display(df.head(5))

In [0]:
# Imports
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, regexp_replace, to_timestamp, year, month, dayofmonth, hour, dayofweek
from pyspark.sql.types import StringType

spark = SparkSession.builder.getOrCreate()

# Pad
train_path = "dbfs:/Workspace/Repos/20076436@student.hhs.nl/MLops-City/Metro_Interstate_Traffic_Volume_train.csv"
test_path  = "dbfs:/Workspace/Repos/20076436@student.hhs.nl/MLops-City/Metro_Interstate_Traffic_Volume_test.csv"
# functie voor schoonmaken
def clean_df(df):

    # 1. duplicaten verwijderen
    df = df.dropDuplicates()

    # 2. datum omzetten
    df = df.withColumn("date_time", to_timestamp(col("date_time"), "yyyy-MM-dd HH:mm:ss"))

    # 3. string kolommen opschonen
    string_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]
    for c in string_cols:
        df = df.withColumn(c, trim(col(c)))
        df = df.withColumn(c, regexp_replace(col(c), r"\s+", " "))

    # 4. numerieke kolommen bepalen
    numeric_types = ["int", "bigint", "double", "float", "long", "short", "decimal"]
    numeric_cols = [f.name for f in df.schema.fields if f.dataType.simpleString() in numeric_types]

    # 5. missende numerieke waardes vullen (mediaan)
    for c in numeric_cols:
        median_value = df.approxQuantile(c, [0.5], 0.01)[0]
        df = df.fillna({c: median_value})

    # 6. missende string waardes vullen
    for c in string_cols:
        df = df.fillna({c: "Unknown"})

    # 7. datum features maken
    df = (
        df
        .withColumn("year", year(col("date_time")))
        .withColumn("month", month(col("date_time")))
        .withColumn("day", dayofmonth(col("date_time")))
        .withColumn("hour", hour(col("date_time")))
        .withColumn("dayofweek", dayofweek(col("date_time")))
    )

    return df


# Alles in 1 lijn uitvoeren (dan wordt neiwue df een keer opgeslagen)
df_train = clean_df(
    spark.read.option("header", True).option("inferSchema", True).csv(train_path)
)

df_test = clean_df(
    spark.read.option("header", True).option("inferSchema", True).csv(test_path)
)

# Resultaat bekijken
display(df_train.limit(5))
display(df_test.limit(5))